In [ ]:
import os
import glob
import sys
import warnings
import importlib.util
import subprocess

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Python version:", sys.version)
print("\nFiles in current directory:")
for f in sorted(os.listdir(".")):
    print(f)

print("\nCSV files found:")
for f in sorted(glob.glob("*.csv")):
    print(f)

In [ ]:
import os, sys, subprocess, importlib.util

for pkg in ["catboost", "lightgbm"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"{pkg} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"{pkg} is already installed.")

GPU_AVAILABLE = False
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if out.returncode == 0:
        GPU_AVAILABLE = True
        print("\nGPU detected.")
    else:
        print("\nNo GPU detected. Running on CPU.")
except FileNotFoundError:
    print("\nNo nvidia-smi on PATH. Running on CPU.")

print("GPU_AVAILABLE =", GPU_AVAILABLE)

In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping

print("CatBoost and LightGBM imported.")
print("GPU available:", GPU_AVAILABLE)

In [ ]:
def find_first_file(patterns):
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[0]
    return None

train_path = find_first_file(["train.csv", "*train*.csv", "*Train*.csv"])
test_path  = find_first_file(["test.csv", "*test*.csv", "*Test*.csv"])
sample_path = find_first_file([
    "sample_submission.csv", "*sample*submission*.csv", "*sample*.csv"
])

print("Selected train file:", train_path)
print("Selected test file:", test_path)
print("Selected sample submission file:", sample_path)

if train_path is None: raise FileNotFoundError("Could not find the training CSV file.")
if test_path  is None: raise FileNotFoundError("Could not find the test CSV file.")
if sample_path is None: raise FileNotFoundError("Could not find the sample submission CSV file.")

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_sub.shape)

print("\nTrain head:")
print(train.head())

print("\nTest head:")
print(test.head())

print("\nSample submission head:")
print(sample_sub.head())

In [ ]:
print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Sample submission columns:", sample_sub.columns.tolist())

target_col = "Will_Buy_EV"

print("\nTarget column in train:", target_col in train.columns)
print("Target column in test:", target_col in test.columns)

if target_col in train.columns:
    print("\nTarget value counts:")
    print(train[target_col].value_counts(dropna=False))
    print("\nTarget percentage:")
    print(train[target_col].value_counts(normalize=True, dropna=False) * 100)

print("\nMissing values in train (top 20):")
print(train.isna().sum().sort_values(ascending=False).head(20))

print("\nMissing values in test (top 20):")
print(test.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
target_col = "Will_Buy_EV"

if target_col not in train.columns:
    raise ValueError(f"Target column '{target_col}' was not found in train.")

train[target_col] = train[target_col].astype(str).str.strip().str.lower()

valid_targets = {"yes", "no"}
unknown_targets = sorted(set(train[target_col].unique()) - valid_targets)
if unknown_targets:
    raise ValueError(f"Unknown target values found: {unknown_targets}")

y = train[target_col].map({"yes": 1, "no": 0})
if y.isna().any():
    raise ValueError("Target mapping failed. Check target values.")
y = y.astype(int)

print("Target mapping complete.")
print("y value counts:")
print(y.value_counts())
print("\ny unique values:", sorted(y.unique().tolist()))

In [ ]:
id_col = "id"
if id_col not in train.columns:
    id_col = train.columns[0]
    print(f"'id' not found in train. Using first train column as id: {id_col}")
else:
    print("Using id column from train:", id_col)

if id_col not in test.columns:
    test_id_col = test.columns[0]
    print(f"'{id_col}' not found in test. Using first test column as test id: {test_id_col}")
else:
    test_id_col = id_col

base_cat_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Range_Anxiety_Level",
    "Home_Charging_Possible",
    "Subsidy_Available"
]

cat_features = []
for c in base_cat_features:
    if c in train.columns and c in test.columns:
        cat_features.append(c)
    else:
        print(f"Expected categorical column missing, skipping: {c}")

# Auto-detect other object columns
for c in train.columns:
    if c in [id_col, target_col]:
        continue
    if c in test.columns and (
        train[c].dtype == "object" or str(train[c].dtype) == "category"
    ):
        if c not in cat_features:
            cat_features.append(c)

print("\nBase categorical features:", cat_features)

for c in cat_features:
    train[c] = train[c].astype(str).str.strip().str.lower()
    test[c]  = test[c].astype(str).str.strip().str.lower()
    train[c] = train[c].replace({"nan": "unknown", "none": "unknown", "": "unknown"})
    test[c]  = test[c].replace({"nan": "unknown", "none": "unknown", "": "unknown"})
    if train[c].isna().any() or test[c].isna().any():
        raise ValueError(f"Missing categorical values found in column: {c}")

print("\nBase categorical columns normalized.")

In [ ]:
def add_features(df):
    df = df.copy()

    # Charging aggregates
    df["charging_total"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    df["charging_diff"]  = df["Charging_Stations_Near_Home"] - df["Charging_Stations_Near_Work"]
    df["charging_min"]   = df[["Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]].min(axis=1)
    df["charging_max"]   = df[["Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]].max(axis=1)
    df["charging_zero"]  = (
        (df["Charging_Stations_Near_Home"] == 0) &
        (df["Charging_Stations_Near_Work"] == 0)
    ).astype(int)

    # Ratio features
    df["income_per_car"]      = df["Annual_Income_USD"] / (df["Number_of_Cars_Owned"] + 1)
    df["commute_per_car"]     = df["Daily_Commute_km"] / (df["Number_of_Cars_Owned"] + 1)
    df["charging_per_commute"] = df["charging_total"] / (df["Daily_Commute_km"] + 1)
    df["income_per_age"]      = df["Annual_Income_USD"] / (df["Age"] + 1)

    # Ordinal mapping for range anxiety
    range_map = {"low": 0, "medium": 1, "high": 2}
    df["range_anxiety_ord"] = df["Range_Anxiety_Level"].map(range_map).fillna(0).astype(int)

    # Simple interactions
    df["income_x_env"]   = df["Annual_Income_USD"] * df["Environmental_Concern_Level"]
    df["commute_x_range"] = df["Daily_Commute_km"] * df["range_anxiety_ord"]

    # Special-value flags (these values look like missing-value codes)
    df["income_is_min"]  = (df["Annual_Income_USD"] == 30000).astype(int)
    df["commute_is_5"]   = (df["Daily_Commute_km"] == 5.0).astype(int)

    # Categorical combinations
    df["home_subsidy"]  = df["Home_Charging_Possible"] + "_" + df["Subsidy_Available"]
    df["city_car"]      = df["City_Type"] + "_" + df["Current_Car_Type"]
    df["gender_city"]   = df["Gender"] + "_" + df["City_Type"]
    df["home_city"]     = df["Home_Charging_Possible"] + "_" + df["City_Type"]
    df["subsidy_range"] = df["Subsidy_Available"] + "_" + df["Range_Anxiety_Level"]
    df["env_range"]     = df["Environmental_Concern_Level"].astype(str) + "_" + df["Range_Anxiety_Level"]

    return df

train = add_features(train)
test  = add_features(test)

new_cat_features = [
    "home_subsidy", "city_car", "gender_city",
    "home_city", "subsidy_range", "env_range"
]
for c in new_cat_features:
    train[c] = train[c].astype(str)
    test[c]  = test[c].astype(str)

cat_features = cat_features + new_cat_features

print("Feature engineering complete.")
print("Total categorical features:", len(cat_features))
print(cat_features)

In [ ]:
feature_cols = [c for c in train.columns if c not in [id_col, target_col]]

missing_in_test = [c for c in feature_cols if c not in test.columns]
if missing_in_test:
    raise ValueError(f"Missing feature columns in test: {missing_in_test}")

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

for c in feature_cols:
    if c in cat_features:
        X[c] = X[c].astype(str).fillna("unknown")
        X_test[c] = X_test[c].astype(str).fillna("unknown")
    else:
        if X[c].isna().any() or X_test[c].isna().any():
            median_value = X[c].median()
            X[c] = X[c].fillna(median_value)
            X_test[c] = X_test[c].fillna(median_value)

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", int(X.isna().sum().sum()))
print("Missing in X_test:", int(X_test.isna().sum().sum()))
print("Feature columns count:", len(feature_cols))

In [ ]:
# LightGBM needs pandas 'category' dtype for categorical features.
# Categories must be identical in X and X_test.
X_lgb = X.copy()
X_test_lgb = X_test.copy()

for c in cat_features:
    categories = sorted(set(X[c].astype(str)) | set(X_test[c].astype(str)))
    X_lgb[c] = pd.Categorical(X[c].astype(str), categories=categories)
    X_test_lgb[c] = pd.Categorical(X_test[c].astype(str), categories=categories)

print("LightGBM-ready categorical columns prepared.")
print("Example categories for 'city_car':")
print(list(X_lgb['city_car'].cat.categories)[:10])

In [ ]:
N_SPLITS = 5   # was 10, faster on CPU. Set to 10 on Kaggle GPU.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

cat_params = dict(
    iterations=1500,          # was 5000
    learning_rate=0.05,       # was 0.03
    depth=6,
    l2_leaf_reg=5.0,
    loss_function="Logloss",
    eval_metric="AUC",
    od_type="Iter",
    od_wait=150,
    auto_class_weights=None,
    max_ctr_complexity=4,
    one_hot_max_size=10,
    random_seed=42,
    verbose=200,
)

if GPU_AVAILABLE:
    cat_params["task_type"] = "GPU"
    cat_params["devices"] = "0"
    cat_params["bootstrap_type"] = "Bernoulli"
    cat_params["subsample"] = 0.8
    cat_params["border_count"] = 128
else:
    cat_params["thread_count"] = -1

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n===== CatBoost Fold {fold}/{N_SPLITS} =====")
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_va, y_va),
        use_best_model=True
    )

    oof_cat[va_idx] = model.predict_proba(X_va)[:, 1]
    test_cat += model.predict_proba(X_test)[:, 1] / N_SPLITS

    print(f"Fold {fold} CatBoost AUC: {roc_auc_score(y_va, oof_cat[va_idx]):.5f}")

    # Save after EVERY fold so a crash does not kill progress
    np.save("oof_cat.npy", oof_cat)
    np.save("test_cat.npy", test_cat)

cat_oof_auc = roc_auc_score(y, oof_cat)
print(f"\nCatBoost OOF AUC: {cat_oof_auc:.5f}")

In [ ]:
lgb_params = dict(
    objective="binary",
    metric="auc",
    n_estimators=1500,       # was 5000
    learning_rate=0.05,      # was 0.03
    num_leaves=63,
    max_depth=-1,
    min_child_samples=30,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l1=0.0,
    lambda_l2=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lgb, y), 1):
    print(f"\n===== LightGBM Fold {fold}/{N_SPLITS} =====")
    X_tr, X_va = X_lgb.iloc[tr_idx], X_lgb.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        categorical_feature=cat_features,
        callbacks=[early_stopping(150, verbose=False)]
    )

    oof_lgb[va_idx] = model.predict_proba(X_va)[:, 1]
    test_lgb += model.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

    print(f"Fold {fold} LightGBM AUC: {roc_auc_score(y_va, oof_lgb[va_idx]):.5f}")

    np.save("oof_lgb.npy", oof_lgb)
    np.save("test_lgb.npy", test_lgb)

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f"\nLightGBM OOF AUC: {lgb_oof_auc:.5f}")

In [ ]:
import os
if os.path.exists("oof_cat.npy") and os.path.exists("test_cat.npy"):
    oof_cat = np.load("oof_cat.npy")
    test_cat = np.load("test_cat.npy")
    print("Loaded CatBoost OOF from disk.")
if os.path.exists("oof_lgb.npy") and os.path.exists("test_lgb.npy"):
    oof_lgb = np.load("oof_lgb.npy")
    test_lgb = np.load("test_lgb.npy")
    print("Loaded LightGBM OOF from disk.")

In [ ]:
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

# Recompute in case this cell runs after a kernel restart
cat_oof_auc = roc_auc_score(y, oof_cat)
lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f"CatBoost OOF AUC: {cat_oof_auc:.5f}")
print(f"LightGBM OOF AUC: {lgb_oof_auc:.5f}")

oof_cat_rank = rankdata(oof_cat) / len(oof_cat)
oof_lgb_rank = rankdata(oof_lgb) / len(oof_lgb)

def neg_auc(w):
    w = np.clip(w, 0, 1)
    s = w.sum()
    if s == 0:
        return 0.0
    w = w / s
    blend = w[0] * oof_cat_rank + w[1] * oof_lgb_rank
    return -roc_auc_score(y, blend)

res = minimize(
    neg_auc,
    x0=[0.5, 0.5],
    method="Nelder-Mead",
    options={"xatol": 1e-4, "fatol": 1e-6}
)

best_w = np.clip(res.x, 0, 1)
best_w = best_w / best_w.sum()

print("\nOptimal blend weights:")
print(f"  CatBoost : {best_w[0]:.4f}")
print(f"  LightGBM : {best_w[1]:.4f}")

blend_oof = best_w[0] * oof_cat_rank + best_w[1] * oof_lgb_rank
blend_oof_auc = roc_auc_score(y, blend_oof)

print(f"\nBlend OOF AUC: {blend_oof_auc:.5f}")

test_cat_rank = rankdata(test_cat) / len(test_cat)
test_lgb_rank = rankdata(test_lgb) / len(test_lgb)

final_test_pred = best_w[0] * test_cat_rank + best_w[1] * test_lgb_rank
print("\nFinal blended test predictions ready. Shape:", final_test_pred.shape)

In [ ]:
print("Sample submission head:")
print(sample_sub.head())
print("\nSample submission columns:", sample_sub.columns.tolist())
print("Sample submission shape:", sample_sub.shape)
print("\nSample submission dtypes:")
print(sample_sub.dtypes)

if len(sample_sub.columns) != 2:
    raise ValueError(
        "Expected sample_submission.csv to have exactly 2 columns: id and target."
    )

submission_id_col = sample_sub.columns[0]
submission_target_col = sample_sub.columns[1]

print("\nSubmission id column:", submission_id_col)
print("Submission target column:", submission_target_col)

In [ ]:
test_ids = test[test_id_col].values

pred_df = pd.DataFrame({
    test_id_col: test_ids,
    submission_target_col: final_test_pred
})

submission = sample_sub[[submission_id_col]].merge(
    pred_df,
    left_on=submission_id_col,
    right_on=test_id_col,
    how="left"
)

if test_id_col != submission_id_col and test_id_col in submission.columns:
    submission = submission.drop(columns=[test_id_col])

submission = submission[[submission_id_col, submission_target_col]]

print("Submission head:")
print(submission.head())
print("\nSubmission shape:", submission.shape)
print("Missing predictions:", int(submission[submission_target_col].isna().sum()))

In [ ]:
print("Checking target mapping...")
if not set(y.unique()).issubset({0, 1}):
    raise ValueError("Target mapping is incorrect.")
print("Target mapping OK.")

print("\nChecking missing values...")
print("X missing:", int(X.isna().sum().sum()))
print("X_test missing:", int(X_test.isna().sum().sum()))
print("Submission missing target:", int(submission[submission_target_col].isna().sum()))

if submission[submission_target_col].isna().any():
    raise ValueError("Submission contains missing predictions.")

print("\nChecking unknown categorical values in test vs train...")
for c in cat_features:
    train_values = set(train[c].astype(str).unique())
    test_values  = set(test[c].astype(str).unique())
    unknown = sorted(test_values - train_values)
    if unknown:
        print(f"{c}: {len(unknown)} unknown values. Examples: {unknown[:5]}")
    else:
        print(f"{c}: no unknown categorical values.")

print("\nChecking submission row count...")
print("Submission rows:", len(submission))
print("Sample submission rows:", len(sample_sub))
if len(submission) != len(sample_sub):
    raise ValueError("Submission row count does not match sample_submission.csv.")

print("\nChecking submission column names...")
print("Submission columns:", submission.columns.tolist())
print("Expected columns:", sample_sub.columns.tolist())
if list(submission.columns) != list(sample_sub.columns):
    raise ValueError("Submission column names do not match sample_submission.csv.")

print("\nAll validation checks passed.")

In [ ]:
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
print("Final submission shape:", submission.shape)
print("\nFinal submission head:")
print(submission.head())

In [ ]:
submission_check = pd.read_csv("submission.csv")
print("Reloaded submission.csv shape:", submission_check.shape)
print("\nReloaded columns:", submission_check.columns.tolist())
print("\nReloaded head:")
print(submission_check.head())
print("\nReloaded missing values:")
print(submission_check.isna().sum())
print("\nPrediction distribution:")
print(submission_check[submission_target_col].describe())